# Build Silver

---

## Imports

In [4]:
import sys
sys.path.insert(0, "../src")

from nlst_pathology.config import PATHOLOGY_ROOT
from nlst_pathology.discovery import list_cancer_types, list_batches, get_batch
from nlst_pathology.catalog import build_metadata_catalog, save_catalog_to_silver

import polars as pl
import pandas as pd
from nlst_pathology.config import SILVER_PATHOLOGY_CATALOG

---

## Listar batches disponíveis na Bronze

In [2]:
print("Cancer types:", list_cancer_types())
print("Batches:", list_batches())

Cancer types: ['2-1-Adenocarcinoma', '2-2-Squamous-Cell-Carcinoma', '2-3-Bronchio-Alveolar-Carcinoma', '2-4-Other-Lung-Cancers']
Batches: ['batch1', 'batch2', 'batch3', 'batch4', 'batch5', 'batch6', 'batch7', 'batch8', 'batch9']


---

## Construir catálogo completo

In [3]:
batches = [get_batch(b) for b in list_batches()]
print(f"Total de batches: {len(batches)}")

catalog = build_metadata_catalog(batches)
print(f"\nCatálogo: {catalog.shape[0]} slides × {catalog.shape[1]} colunas")
print(catalog.head())

Total de batches: 9


KeyboardInterrupt: 

---

## Salvar na Silver

In [ ]:
save_catalog_to_silver(catalog)

---

## Validar leitura da Silver

In [2]:
df = pl.read_parquet(SILVER_PATHOLOGY_CATALOG)
print(f"Silver pathology: {df.shape[0]} rows × {df.shape[1]} cols")
print(df.schema)

Silver pathology: 1225 rows × 65 cols
Schema({'batch': String, 'cancer_type': String, 'filename': String, 'slide_id': String, 'path': String, 'file_size_bytes': Int64, 'file_size_mb': Float64, 'file_size_gb': Float64, 'width': Int64, 'height': Int64, 'level_count': Int64, 'aperio.AppMag': String, 'aperio.DSR ID': String, 'aperio.Date': String, 'aperio.Filename': String, 'aperio.Focus Offset': String, 'aperio.ICC Profile': String, 'aperio.ImageID': String, 'aperio.Left': String, 'aperio.LineAreaXOffset': String, 'aperio.LineAreaYOffset': String, 'aperio.LineCameraSkew': String, 'aperio.MPP': String, 'aperio.OriginalWidth': String, 'aperio.Originalheight': String, 'aperio.ScanScope ID': String, 'aperio.StripeWidth': String, 'aperio.Time': String, 'aperio.Top': String, 'aperio.User': String, 'openslide.associated.macro.height': String, 'openslide.associated.macro.width': String, 'openslide.associated.thumbnail.height': String, 'openslide.associated.thumbnail.icc-size': String, 'openslide.

In [ ]:
# Patologia
path = pl.read_parquet("../datalake/silver/pathology_metadata.parquet")
print("Pathology — slide_id samples:")
print(path.select("slide_id", "cancer_type").head(10))

# Lung Cancer
lc = pl.read_parquet("../datalake/silver/lung_cancer_metadata.parquet")
print("\nLung Cancer — patient_id samples:")
print(lc.select("patient_id", "PatientID", "cancer_category").unique().head(10))

Pathology — slide_id samples:
shape: (10, 2)
┌──────────┬────────────────────┐
│ slide_id ┆ cancer_type        │
│ ---      ┆ ---                │
│ str      ┆ str                │
╞══════════╪════════════════════╡
│ 10015    ┆ 2-1-Adenocarcinoma │
│ 10037    ┆ 2-1-Adenocarcinoma │
│ 10125    ┆ 2-1-Adenocarcinoma │
│ 10126    ┆ 2-1-Adenocarcinoma │
│ 10127    ┆ 2-1-Adenocarcinoma │
│ 10183    ┆ 2-1-Adenocarcinoma │
│ 10184    ┆ 2-1-Adenocarcinoma │
│ 10378    ┆ 2-1-Adenocarcinoma │
│ 10379    ┆ 2-1-Adenocarcinoma │
│ 10380    ┆ 2-1-Adenocarcinoma │
└──────────┴────────────────────┘

Lung Cancer — patient_id samples:
shape: (10, 3)
┌────────────┬───────────┬─────────────────┐
│ patient_id ┆ PatientID ┆ cancer_category │
│ ---        ┆ ---       ┆ ---             │
│ str        ┆ str       ┆ str             │
╞════════════╪═══════════╪═════════════════╡
│ 127890     ┆ 127890    ┆ screen_detected │
│ 200925     ┆ 200925    ┆ screen_detected │
│ 132386     ┆ 132386    ┆ screen_detected │
│

In [7]:
lc = pl.read_parquet("../datalake/silver/lung_cancer_metadata.parquet")
print(f"Shape: {lc.shape}")
print(f"\nColunas:\n{lc.columns}")
print(f"\nSample:")
print(lc.head(3))

Shape: (666943, 105)

Colunas:
['SpecificCharacterSet', 'ImageType', 'SOPClassUID', 'SOPInstanceUID', 'StudyDate', 'StudyTime', 'AccessionNumber', 'Modality', 'Manufacturer', 'ReferringPhysicianName', 'StationName', 'StudyDescription', 'SeriesDescription', 'ManufacturerModelName', 'PatientName', 'PatientID', 'PatientBirthDate', 'PatientSex', 'OtherPatientIDs', 'ClinicalTrialSponsorName', 'ClinicalTrialProtocolID', 'ClinicalTrialProtocolName', 'ClinicalTrialSiteID', 'ClinicalTrialSiteName', 'ClinicalTrialSubjectID', 'ClinicalTrialTimePointID', 'ClinicalTrialCoordinatingCenterName', 'PatientIdentityRemoved', 'DeidentificationMethod', 'DeidentificationMethodCodeSequence', 'BodyPartExamined', 'SliceThickness', 'KVP', 'DataCollectionDiameter', 'SoftwareVersions', 'ReconstructionDiameter', 'DistanceSourceToDetector', 'DistanceSourceToPatient', 'GantryDetectorTilt', 'TableHeight', 'RotationDirection', 'ExposureTime', 'XRayTubeCurrent', 'Exposure', 'FilterType', 'GeneratorPower', 'FocalSpots',

In [8]:
lc = pl.read_parquet("../datalake/silver/pathology_metadata.parquet")
print(f"Shape: {lc.shape}")
print(f"\nColunas:\n{lc.columns}")
print(f"\nSample:")
print(lc.head(3))

Shape: (1225, 65)

Colunas:
['batch', 'cancer_type', 'filename', 'slide_id', 'path', 'file_size_bytes', 'file_size_mb', 'file_size_gb', 'width', 'height', 'level_count', 'aperio.AppMag', 'aperio.DSR ID', 'aperio.Date', 'aperio.Filename', 'aperio.Focus Offset', 'aperio.ICC Profile', 'aperio.ImageID', 'aperio.Left', 'aperio.LineAreaXOffset', 'aperio.LineAreaYOffset', 'aperio.LineCameraSkew', 'aperio.MPP', 'aperio.OriginalWidth', 'aperio.Originalheight', 'aperio.ScanScope ID', 'aperio.StripeWidth', 'aperio.Time', 'aperio.Top', 'aperio.User', 'openslide.associated.macro.height', 'openslide.associated.macro.width', 'openslide.associated.thumbnail.height', 'openslide.associated.thumbnail.icc-size', 'openslide.associated.thumbnail.width', 'openslide.comment', 'openslide.icc-size', 'openslide.level-count', 'openslide.level[0].downsample', 'openslide.level[0].height', 'openslide.level[0].tile-height', 'openslide.level[0].tile-width', 'openslide.level[0].width', 'openslide.level[1].downsample', 

In [9]:
lc = pl.read_parquet("../datalake/silver/masterfile.parquet")
print(f"Shape: {lc.shape}")
print(f"\nColunas:\n{lc.columns}")
print(f"\nSample:")
print(lc.head(3))

Shape: (1252, 279)

Colunas:
['dataset_version', 'pid', 'image_filename', 'image_filesize', 'lc_order', 'selected_path', 'selected_path_batch', 'block_annotated', 'block_color1', 'block_color2', 'block_color3', 'block_color4', 'block_color5', 'block_dnacore_cnt', 'block_dnacore_orig_cnt', 'block_dnacore_vials', 'block_has_dnacore', 'block_has_tmacore', 'block_slide_cnt', 'block_tmacore_cnt', 'block_tmacore_nontumor_cnt', 'block_tmacore_orig_cnt', 'block_tmacore_tumor_cnt', 'block_worst_class', 'block_worst_grade', 'block_worst_grade_qual', 'roi_NThist_Chronic_bronchitis1', 'roi_NThist_Chronic_bronchitis2', 'roi_NThist_Chronic_bronchitis3', 'roi_NThist_Chronic_bronchitis4', 'roi_NThist_Chronic_bronchitis5', 'roi_NThist_Chronic_bronchitis6', 'roi_NThist_Emphysema1', 'roi_NThist_Emphysema2', 'roi_NThist_Emphysema3', 'roi_NThist_Emphysema4', 'roi_NThist_Emphysema5', 'roi_NThist_Emphysema6', 'roi_NThist_Fibrosis1', 'roi_NThist_Fibrosis2', 'roi_NThist_Fibrosis3', 'roi_NThist_Fibrosis4', 'roi

---

## Exploração do Masterfile

In [5]:
# SAS7BDAT é lido nativamente pelo pandas
masterfile = pd.read_sas(
    "../datalake/bronze/dictionaries/pathology_images_d040722.sas7bdat",
    encoding="latin-1",
)

print(f"Shape: {masterfile.shape}")
print(f"\nColunas: {masterfile.columns.tolist()}")
print(f"\nTipos:\n{masterfile.dtypes}")
print(f"\nPrimeiras linhas:")
print(masterfile.head(10))

Shape: (1252, 279)

Colunas: ['dataset_version', 'pid', 'image_filename', 'image_filesize', 'lc_order', 'selected_path', 'selected_path_batch', 'block_annotated', 'block_color1', 'block_color2', 'block_color3', 'block_color4', 'block_color5', 'block_dnacore_cnt', 'block_dnacore_orig_cnt', 'block_dnacore_vials', 'block_has_dnacore', 'block_has_tmacore', 'block_slide_cnt', 'block_tmacore_cnt', 'block_tmacore_nontumor_cnt', 'block_tmacore_orig_cnt', 'block_tmacore_tumor_cnt', 'block_worst_class', 'block_worst_grade', 'block_worst_grade_qual', 'roi_NThist_Chronic_bronchitis1', 'roi_NThist_Chronic_bronchitis2', 'roi_NThist_Chronic_bronchitis3', 'roi_NThist_Chronic_bronchitis4', 'roi_NThist_Chronic_bronchitis5', 'roi_NThist_Chronic_bronchitis6', 'roi_NThist_Emphysema1', 'roi_NThist_Emphysema2', 'roi_NThist_Emphysema3', 'roi_NThist_Emphysema4', 'roi_NThist_Emphysema5', 'roi_NThist_Emphysema6', 'roi_NThist_Fibrosis1', 'roi_NThist_Fibrosis2', 'roi_NThist_Fibrosis3', 'roi_NThist_Fibrosis4', 'roi